# 02 - Deep Agent com planejamento

No notebook anterior, o deep agent respondeu a uma pergunta direta. Aqui ele recebe uma tarefa com etapas e precisa **planejar antes de executar**.

Fluxo:

```
START → planeja (write_todos) → executa a tarefa → resposta final → END
```

O planejamento usa a ferramenta `write_todos`. Ela cria uma lista de passos e atualiza o status de cada um. Essa ferramenta não vem por padrão no `create_deep_agent`, então o notebook a adiciona com o `TodoListMiddleware`, do LangChain.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. Criando o agente

O `system_prompt` exige a ordem: primeiro o plano com `write_todos`, depois a execução. O `TodoListMiddleware` adiciona a ferramenta `write_todos` ao agente.

In [2]:
agente = create_deep_agent(
    model=llm,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Você é um assistente que resolve tarefas com etapas. "
        "Antes de responder, chame a ferramenta write_todos com um plano de 3 a 5 passos. "
        "Depois de planejar, execute os passos e escreva a resposta final em português."
    ),
)

## 3. Executando

A função `executar` roda a tarefa e mostra as mensagens na ordem em que aconteceram. A primeira chamada de `write_todos` mostra o plano. Chamadas seguintes atualizam o status dos passos.

In [3]:
def executar(tarefa: str) -> dict:
    estado = agente.invoke({"messages": [HumanMessage(content=tarefa)]})

    linhas = [f"**Tarefa:** {tarefa}", ""]
    plano_mostrado = False
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                if chamada["name"] == "write_todos" and not plano_mostrado:
                    linhas.append("**Plano:**")
                    for item in chamada["args"].get("todos", []):
                        linhas.append(f"- {item.get('content', item)}")
                    linhas.append("")
                    plano_mostrado = True
                elif chamada["name"] == "write_todos":
                    linhas.append("→ `write_todos` (atualiza o status dos passos)")
                else:
                    linhas.append(f"→ `{chamada['name']}`")
        elif isinstance(mensagem, AIMessage) and mensagem.content:
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))
    return estado


estado = executar(
    "Monte um roteiro de estudo de 3 dias para aprender LangGraph, com um tema por dia."
)

**Tarefa:** Monte um roteiro de estudo de 3 dias para aprender LangGraph, com um tema por dia.

**Plano:**
- Definir objetivos, público-alvo e temas por dia para o roteiro de 3 dias
- Criar roteiro detalhado para Dia 1 (introdução, conceitos e setup)
- Criar roteiro detalhado para Dia 2 (construção de fluxos, componentes e testes)
- Criar roteiro detalhado para Dia 3 (integrações, deploy e projeto final)

→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)

**Resposta:** Plano de estudo de 3 dias para aprender LangGraph

Objetivo geral
- Em 3 dias ganhar conhecimento prático suficiente para projetar, implementar e testar fluxos básicos e integrados usando LangGraph, e ter um mini-projeto funcional ao final.

Público-alvo
- Desenvolvedores com noções básicas de Python e de modelos LLM/embeddings (outras linguagens adaptáveis), que querem construir pipelines orientados a grafos para LLMs e integrações.

Pré-requisitos
- Python 3.8+ instalado.
- Conhecimentos básicos de chamadas a APIs (ex.: OpenAI) e de contêineres/HTTP é útil.
- Ter chaves de API necessárias (ex.: OPENAI_API_KEY) se for usar LLMs remotos.
- Instalar a biblioteca oficial do LangGraph (verifique o nome do pacote na documentação; um comando comum é: pip install langgraph) e ferramentas auxiliares (requests, uvicorn, fastapi, etc.).

Sugestão de carga horária por dia
- 3 a 5 horas por dia (leitura + hands-on). Ajuste conforme sua disponibilidade.

Dia 1 — Tema: Fundamentos, arquitetura e setup (Objetivo: entender conceitos e executar o primeiro fluxo)
- Objetivos específicos:
  - Compreender o que é LangGraph e quando usá-lo.
  - Conhecer conceitos principais: nós (nodes), arestas (edges), fluxos/graphs, runtime, entradas/saídas, estado.
  - Instalar, configurar e rodar o quickstart.
- Cronograma sugerido (4 horas):
  1. 30–45 min — Leitura introdutória
     - Leia a visão geral na documentação oficial e o README do GitHub.
     - Entenda casos de uso (orquestração de prompts, pipelines híbridos LLM+API, controle de fluxo).
  2. 30 min — Instalação e setup
     - pip install langgraph (confirme nome na docs).
     - Configure variáveis de ambiente (ex.: export OPENAI_API_KEY=...).
  3. 60–90 min — Quickstart hands-on
     - Rode um exemplo “hello world” do quickstart.
     - Crie um grafo simples: entrada de texto -> nó LLM que gera resposta -> nó de saída que grava em arquivo/print.
     - Execute e observe logs/outputs.
  4. 30–45 min — Experimentos básicos
     - Mude prompt, adicione parâmetros (temperature, max_tokens).
     - Experimente salvar/ler estado entre nós (se aplicável).
- Exercícios práticos (entregáveis):
  - Entregar um script simples que recebe um prompt e retorna a resposta via LangGraph.
  - Documentar 3 diferenças entre usar LangGraph vs um script linear de chamadas LLM.
- Dicas:
  - Ative logging para ver o fluxo de execução.
  - Use um LLM local (se disponível) para testes offline ou limite custos usando mocks.

Dia 2 — Tema: Componentes, controle de fluxo e testes (Objetivo: construir fluxos não-triviais e testá-los)
- Objetivos específicos:
  - Aprender a criar componentes/nós customizados.
  - Implementar branching/condicionais, loops e tratamento de erros.
  - Testar e fazer debug de fluxos.
- Cronograma sugerido (4–5 horas):
  1. 30–45 min — Revisão de conceitos avançados
     - Nodes customizados, parâmetros, schemas de entrada/saída, tipagem.
  2. 60–90 min — Implementação hands-on
     - Construa um pipeline com vários nós: pré-processamento -> LLM -> chamada a API externa (ex.: Wikipedia API) -> pós-processamento.
     - Implemente uma ramificação: se a confiança/score for abaixo de X, peça reescrita ao LLM; caso contrário, finalize.
  3. 45–60 min — Tratamento de erros e retries
     - Adicione retries, timeouts, e manejo de exceções em nós críticos.
  4. 30–45 min — Testes automatizados
     - Escreva testes unitários para nós individuais (mockando chamadas LLM/API).
     - Teste de integração para o grafo completo (pode usar fixtures que simulam respostas).
- Exercícios práticos (entregáveis):
  - Pipeline que recebe pergunta do usuário, realiza busca em API externa, compõe resposta com LLM e decide por re-chamada caso a resposta não tenha evidência suficiente.
  - Conjunto de tests (pytest) que cobrem pelo menos 3 cenários (sucesso, fallback, erro externo).
- Dicas:
  - Use mocks para evitar custos com chamadas reais em testes.
  - Registre métricas simples (latência por nó, contagem de retries) para ajudar no debug.

Dia 3 — Tema: Integrações, deploy e projeto final (Objetivo: integrar, empacotar e colocar em produção um mini-projeto)
- Objetivos específicos:
  - Integrar LangGraph com armazenamento/embeddings e com uma interface externa.
  - Containerizar e preparar deploy simples.
  - Finalizar um mini-projeto que combine tudo aprendido.
- Cronograma sugerido (4–5 horas):
  1. 30–45 min — Planejamento do mini-projeto
     - Defina o projeto: ex.: chatbot de FAQ com busca por embeddings + LangGraph para orquestração e geração final.
     - Tecnologias: LangGraph + vector DB (ex.: FAISS, Pinecone, Weaviate) + embeddings + FastAPI frontend.
  2. 90–120 min — Implementação da integração
     - Adicione nó que consulta vetor DB (ou FAISS local).
     - Use embeddings para recuperar documentos relevantes e passe ao LLM para geração.
     - Construa endpoint HTTP simples (FastAPI/Flask) que aciona o graf o e retorna resposta ao usuário.
  3. 45–60 min — Empacotamento e deploy
     - Dockerize a aplicação: Dockerfile com dependências.
     - Deploy simples em um serviço (Render, Heroku, Vercel) ou execução em VM/container local.
  4. 30 min — Testes finais e melhorias
     - Teste com exemplos reais, monitore latência, ajuste prompts e fallback.
- Projeto final (entrega sugerida):
  - Um chatbot básico que:
    - Recebe pergunta via HTTP.
    - Busca documentos relevantes no vector store.
    - Orquestra com LangGraph: recuperar ~3 trechos, gerar resposta via LLM, aplicar validação e fallback se necessário.
    - Endpoint pronto e containerizado.
- Extensões (stretch goals):
  - Autenticação, logs estruturados, dashboard de métricas, A/B testing de prompts.
  - Integrar com frontend simples (HTML/JS) para interação em tempo real.
- Dicas de deployment e produção:
  - Separe credenciais por variáveis de ambiente e secrets manager.
  - Monitore uso de tokens/custos.
  - Considere rate-limiting e políticas de retry.

Checklist final (o que ter ao fim dos 3 dias)
- Você terá:
  - Um entendimento sólido da arquitetura de LangGraph.
  - 1–2 exemplos práticos: quickstart e um pipeline multi-nó.
  - Testes básicos cobrindo lógica chave.
  - Um mini-projeto integrado com vector store e endpoint HTTP, pronto para deploy.

Recursos recomendados (onde procurar)
- Documentação oficial do LangGraph (README e seção Quickstart do repositório GitHub).
- Exemplos do repositório oficial (folders “examples”).
- Tutoriais/guia de integração com embeddings e vector DB (docs do provedor escolhido).
- Ferramentas úteis: pytest (testes), Docker (empacotamento), FastAPI/uvicorn (API), FAISS/Pinecone/Weaviate (vector DB).
(Nota: confirme nomes de pacotes e comandos na documentação oficial do LangGraph antes de instalar.)

Se quiser, eu posso:
- Gerar scripts de exemplo prontos (quickstart, pipeline do Dia 2, Dockerfile e FastAPI) para você executar.
- Adaptar o plano para menos/more tempo por dia, ou para outra linguagem além de Python.

Quer que eu comece gerando um exemplo de projeto do Dia 3 (esqueleto FastAPI + LangGraph + FAISS) ou um quickstart para o Dia 1?

## 4. Conferindo a ordem

Para confirmar que o planejamento veio antes da execução, listamos as chamadas de ferramenta em ordem. A primeira deve ser `write_todos`.

In [4]:
ordem = []
for mensagem in estado["messages"]:
    if isinstance(mensagem, AIMessage):
        ordem.extend(chamada["name"] for chamada in mensagem.tool_calls)

print(" → ".join(ordem) if ordem else "nenhuma ferramenta chamada")
print("Planejou primeiro:", bool(ordem) and ordem[0] == "write_todos")

write_todos → write_todos → write_todos → write_todos → write_todos
Planejou primeiro: True


## Observações

- Sem o `TodoListMiddleware`, o agente não tem a ferramenta `write_todos`. O prompt sozinho não basta.
- O plano é só uma lista de passos. Quem garante a ordem é o `system_prompt`. Por isso a célula de conferência verifica a ordem das chamadas.
- Esta versão planeja e responde em sequência. Ferramentas e subagentes ficam para os próximos notebooks.